In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import os, glob, random, time
import kagglehub

In [2]:
seed = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = '/kaggle/input/competitions/apple-disease-classification-2026-t-2/data'
img_size = 224
batch_size = 32
epochs = 12

In [3]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print(f"The training is running on: {device}")

The training is running on: cuda


In [4]:
train_folder = os.path.join(root, 'train')
test_folder = os.path.join(root,'test')
classes = sorted(os.listdir(train_folder))

In [5]:
rows = []
for class_ in classes:
    for filepath in glob.glob(os.path.join(train_folder,class_,"*")):
        rows.append({'filepath': filepath, 'label': class_})
df = pd.DataFrame(rows)
df.sample(5)

,filepath,label
5562,/kaggle/input/competitions/apple-disease-class...,healthy
1188,/kaggle/input/competitions/apple-disease-class...,apple_scab
5352,/kaggle/input/competitions/apple-disease-class...,healthy
3131,/kaggle/input/competitions/apple-disease-class...,black_rot
743,/kaggle/input/competitions/apple-disease-class...,apple_scab


In [6]:
df.shape

(6771, 2)

In [7]:
df.label.value_counts().to_frame()

,count
label,
apple_scab,1757
healthy,1750
black_rot,1731
cedar_apple_rust,1533


In [8]:
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}
df['label_idx'] = df['label'].map(label2idx)

In [9]:
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label_idx'], random_state=seed)
print(train_df.shape, val_df.shape)

(5755, 3) (1016, 3)


In [10]:
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [11]:
eval_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [12]:
class AppleDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        img = self.transform(img)
        if self.is_test:
            return img, row['id']
        return img, row['label_idx']
    

In [13]:
train_ds = AppleDataset(train_df, train_tfms)
val_ds = AppleDataset(val_df, eval_tfms)

In [14]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

In [15]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 223MB/s]


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [17]:
best_f1 = 0.0
best_state = None

for epoch in range(epochs):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    scheduler.step()
    train_loss = total_loss / len(train_ds)
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f'Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | val_weighted_f1 {val_f1:.4f} | time {time.time()-t0:.1f}s')

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

Epoch 1/12 | train_loss 0.0912 | val_weighted_f1 0.9951 | time 36.4s
Epoch 2/12 | train_loss 0.0237 | val_weighted_f1 0.9543 | time 24.5s
Epoch 3/12 | train_loss 0.0345 | val_weighted_f1 0.9951 | time 24.3s
Epoch 4/12 | train_loss 0.0126 | val_weighted_f1 0.9971 | time 24.8s
Epoch 5/12 | train_loss 0.0140 | val_weighted_f1 0.9990 | time 24.7s
Epoch 6/12 | train_loss 0.0049 | val_weighted_f1 1.0000 | time 25.0s
Epoch 7/12 | train_loss 0.0025 | val_weighted_f1 1.0000 | time 24.8s
Epoch 8/12 | train_loss 0.0028 | val_weighted_f1 1.0000 | time 24.5s
Epoch 9/12 | train_loss 0.0017 | val_weighted_f1 1.0000 | time 24.6s
Epoch 10/12 | train_loss 0.0019 | val_weighted_f1 1.0000 | time 24.9s
Epoch 11/12 | train_loss 0.0014 | val_weighted_f1 1.0000 | time 24.5s
Epoch 12/12 | train_loss 0.0014 | val_weighted_f1 1.0000 | time 24.8s


In [18]:
print('Best val weighted F1:', best_f1)
model.load_state_dict(best_state)

Best val weighted F1: 1.0


<All keys matched successfully>

In [19]:
test_files = sorted(glob.glob(os.path.join(test_folder, '*')))
test_df = pd.DataFrame({
    'filepath': test_files,
    'id': [os.path.splitext(os.path.basename(f))[0] for f in test_files]
})
print(test_df.shape)

(1000, 2)


In [20]:
test_ds = AppleDataset(test_df, eval_tfms, is_test=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

In [21]:
model.eval()
pred_ids, pred_labels = [], []
with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        preds = out.argmax(dim=1).cpu().numpy()
        pred_ids.extend(ids)
        pred_labels.extend([idx2label[p] for p in preds])

In [22]:
submission = pd.DataFrame({'id': pred_ids, 'target': pred_labels})
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)
submission.head()

,id,target
0,img000,apple_scab
1,img001,black_rot
2,img002,black_rot
3,img003,black_rot
4,img004,black_rot
